# DDG Regularizer – Colab Runner
**Runtime → Change runtime type → T4 GPU** before running.

Cells run top-to-bottom. Each section is independent so you can re-run from any cell.

## 1. Install dependencies

In [ ]:
import subprocess, sys

# PyTorch Geometric wheel set matched to the Colab-default torch version
import torch
torch_ver = torch.__version__.split('+')[0]   # e.g. '2.3.0'
cuda_tag  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
print(f'torch {torch_ver}  cuda_tag={cuda_tag}')

pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch-scatter', 'torch-sparse', 'torch-cluster',
                'torch-spline-conv', 'torch-geometric',
                '-f', pyg_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'scipy', 'tqdm', 'pandas'], check=True)
print('All dependencies installed.')

## 2. Clone the repo

In [ ]:
import os

REPO = 'https://github.com/msvkvprasad1976/claude.git'
BRANCH = 'claude/run-all-files-lo8qo4'

if not os.path.isdir('claude'):
    !git clone --branch {BRANCH} {REPO}
else:
    !git -C claude pull origin {BRANCH}

os.chdir('claude')
print('Working directory:', os.getcwd())
!ls

## 3. Smoke test (no GPU / no dataset needed, ~15 s)

In [ ]:
!python SMOKE_TEST.py

## 4a. Fast smoke training – ModelNet40 (3 epochs, 200 samples)
Downloads ~500 MB of ModelNet40 on first run. Subsequent runs use the cache.

In [ ]:
!python train_modelnet40.py \
    --epochs 3 \
    --limit 200 \
    --seed 0 \
    --num_workers 2 \
    --batch_size 16 \
    --weighting cotangent \
    --baseline ddg

## 4b. Fast smoke training – QM9 (3 epochs, 2000 molecules)
Downloads ~130 MB of QM9 on first run.

In [ ]:
!python train_qm9.py \
    --epochs 3 \
    --limit 2000 \
    --seed 0 \
    --num_workers 2 \
    --batch_size 64 \
    --baseline ddg

## 5. Full protocol – 5 seeds × all baselines (hours)
**Only run after the smoke runs above pass.**  
This writes per-run JSON logs under `./outputs/`.

In [ ]:
# Uncomment to launch:
# !bash run_all.sh

## 6. Aggregate results → summary table

In [ ]:
!python aggregate_results.py
print('\n--- summary.md ---')
!cat outputs/tables/summary.md